# Milestone-2
This milestone has been created to transition you from classical NLP techiniques to modern, state-of-the-art deep learning architectures. it focuses on familiarizing you with the Hugging Face ecosystem, understanding how attention maechanisms create context-aware representations, and leveraging pre-trained models and zero-shot classification to drastically improve upon you baseline ranking metrics.

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


1. Load **train.csv** using the Huggin Face datasets library ( do not use pandas ). Use the .map() function to create a new column called combined_text that concatenates the prompt and A columns with a space in between. What is the exact character length ( total number of string charactercs using Python's len() function, NOT the number of tokens ) of the combined_text string for the row at index 51 ? Note: We follow zero-indexing here.

In [2]:
from datasets import load_dataset

dataset = load_dataset("csv", data_files="/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")["train"]

def combine_columns(example):
    example["combined_text"] = example["prompt"] + " " + example["A"]
    return example

dataset = dataset.map(combine_columns)

row = dataset[50]

length = len(row["combined_text"])
print(length)

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

294


2. Initialize the bert-base-uncased tokenizer. Look at the tokenizer's configuration properties: What is the exact total vocabulary size (the maximum number of unique subword tokens the model knows) hardcoded into this tokenizer?

In [4]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

print("vocabulary size:", tokenizer.vocab_size)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

vocabulary size: 30522


3. Transformers rely on sepical tokens to understand sentence boundaries. Using the ***bert-base-uncased*** tokenizer from the previous step, extract the exact integer ID assigned to the **[SEP]** (seperator) token.

In [14]:
sep_id = tokenizer.convert_tokens_to_ids("[SEP]")
print("The [SEP] token ID is:", sep_id)

The [SEP] token ID is: 102


4. Using the bert-base-uncased tokenizer, tokenize the entire prompt column of the train dataset simultaneouly. Set ***padding='max_length', turnication=True, max_length=128*** and ***return_tensor='pt'*** ( PyTorch tensor ).

What is the exact geometric shape (dimentions) of the resulting **input_ids** tensor?

In [20]:
prompts = list(dataset['prompt'])
tokenized = tokenizer(
    prompts,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)
print("Shape of input_ids:", tokenized["input_ids"].shape)

Shape of input_ids: torch.Size([2000, 128])


## BERT/RoBERTa Architecture & Attention Mechanisms

5. A standard ***bert-base-uncased*** model has a hidden embedding size of 768 dimentions and uses exactly 12 attention heads in each layer.

   In Transformer architecture, the hidden size is divided equally among the attention heads. What is the exact dimentionality (size) of each individual attention head ?

In [22]:
print(f"Dimentionality of each individual attention head:",768/12)

Dimentionality of each individual attention head: 64.0


6. Load the bert-base-uncased model using AutoModel.from_pretrained(). Tokenize the prompt form row ID 0 using the tokenizer's default setting ( do not apply any manual padding or turnication ). Pass this tokenizer input through the model. Look at the output object.

   What is the exact shape of the last_hidden_state tensor returned ?
   
   *Note: We follow zero-indexing here.*

In [6]:
from transformers import AutoModel

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")

inputs = tokenizer(dataset[0]["prompt"], return_tensors="pt")

outputs = model(**inputs)

print("Shape of last_hidden_state:", outputs.last_hidden_state.shape)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Shape of last_hidden_state: torch.Size([1, 31, 768])


7. Using the ***last_hidden_state*** tensor from the previous question, extract the embedding vector representing the **\[CLS]** token ( which is always the token at index 0 ). What is the sum of the first 5 float values in this **\[CLS]** vector? (Round your answer to 4 decimal places).

In [8]:
cls_vector = outputs.last_hidden_state[0,0,:]
sum_f5 = cls_vector[:5].sum().item()
print("Sum of first 5 values:", round(sum_f5, 4))

Sum of first 5 values: -1.2001


8. Load **bert-base-uncased** with the parameter ***ouput_attentions=True***. Tokenize the exact string "Light-ion fusion is a technique." (ensure you set ***return_tensors='pt'***) and pass it through the model. Extract the attention matrix for the last layer (index -1) and the first attention head (head index 0).

   What is the exact attention weight ( a float value ) that the **\[CLS]** token ( token index 0 ) pays to the word fusion (you will need to find the specific token index for fusion in the input_ids)? (Round your answer to 4 decimal places).

In [10]:
import torch

model = AutoModel.from_pretrained("bert-base-uncased", output_attentions=True)

inputs = tokenizer("Light-ion fusion is a technique.", return_tensors="pt")

outputs = model(**inputs)

attentions = outputs.attentions

last_layer_attention = attentions[-1]

head0_attention = last_layer_attention[0, 0]

fusion_index = (inputs["input_ids"][0]==tokenizer.convert_tokens_to_ids("fusion")).nonzero(as_tuple=True)[0].item()

cls_to_fusion_weight = head0_attention[0, fusion_index].item()

print("Attention weight [CLS] -> fusion:", round(cls_to_fusion_weight, 4))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Attention weight [CLS] -> fusion: 0.1025


## Context-Aware Embeddings
9. Initialize the **sentence-transforms/all-MiniLM-L6-v2** model. Use the model's .encode() method to generate embeddings for both the prompt and Option B for row ID 0. Calculate the cosine similarity between these two vectors specifically using the ***sentence_transformers.util.cos_sim()*** function. What is the resulting similarity score rounded to 4 decimal places? *Note: We follow zero indexing here.*

In [12]:
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

prompt_text = dataset[0]["prompt"]
optionB_text = dataset[0]['B']

emb_prompt = model.encode(prompt_text, convert_to_tensor=True)
emb_optionB = model.encode(optionB_text, convert_to_tensor=True)

similarity_score = util.cos_sim(emb_prompt, emb_optionB).item()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [13]:
print("Cosine similarity:", round(similarity_score, 4))

Cosine similarity: 0.7658


10. Build two complete ranking pipelines evaluating every row in train.csv.

    Pipeline 1: Use the TF-IDF cosine similarity approach from Milestone 1.

    Pipeline 2: Use the **sentence-Transformers/all-MiniLM-L6-v2** model to generate embeddings for the prompt and all five options. Rank options using cosine similarity to form Top-3 predictions.

    First, what is the final MAP@3 score of the **all-MiniLM-L6-v2** pipeline across the entire training set?

    second, count the number of questions for which the corret answer is NOT present in the TF-IDF Top-3 predictions **BUT IS** present in the MiniLM Top-3 predictions. What is the exact resulting count?

In [24]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

prompts = dataset['prompt']
options = [ [row['A'] ,row['B'] ,row['C'] ,row['D'] ,row['E'] ] for row in dataset ]
answers = dataset['answer']

prompts_list = list(dataset["prompt"])
flat_options = [opt for row in options for opt in row]

all_texts = prompts_list + flat_options
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(all_texts)

def tfidf_top3(i):
    p_vec = tfidf_matrix[i]
    o_vec = tfidf_matrix[len(prompts)+i*5 : len(prompts)+(i+1)*5]
    sims = cosine_similarity(p_vec, o_vec)[0]
    return np.argsort(sims)[::-1][:3]

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

def minilm_top3(i):
    emb_p = model.encode(prompts[i], convert_to_tensor=True)
    emb_o = model.encode(options[i], convert_to_tensor=True)
    sims = util.cos_sim(emb_p, emb_o)[0].cpu().numpy()
    return np.argsort(sims)[::-1][:3]

def map3(preds, truth):
    score = 0
    for i, p in enumerate(preds):
        if truth[i] in p:
            rank = list(p).index(truth[i]) + 1
            score += 1.0 / rank
    return score / len(preds)

tfidf_preds = [tfidf_top3(i) for i in range(len(prompts))]
minilm_preds = [minilm_top3(i) for i in range(len(prompts))]

map3_minilm = map3(minilm_preds, answers)

count_rescued = sum(
    (answers[i] not in tfidf_preds[i]) and (answers[i] in minilm_preds[i])
    for i in range(len(prompts))
)

print("MAP@3 (MiniLM):", round(map3_minilm, 4))
print("Rescued count:", count_rescued)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


MAP@3 (MiniLM): 0.0
Rescued count: 0


## Zero-shot calssification concepts

11. initialize the Hugging Face pipeline for **"zero-shot-classification"** (it will default to ***facebook/bart-large-mnli). For the prompt of the 2nd row (index 1). pass Options A, B, and C as the candidate_labels. What is the probability score given to the top-3 randed option? (Round to 4 decimal places).

In [25]:
from transformers import pipeline

classifier = pipeline("zero-shot-classification")

prompt_text = dataset[1]['prompt']

candidate_labels = [dataset[1]['A'], dataset[1]['B'], dataset[1]['C']]

result = classifier(prompt_text, candidate_labels)

for label, score in zip(result["labels"], result["scores"]):
    print(label, round(score, 4))

No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively easy to implement and can be done in an efficient manner, requiring only a vacuum tube, a pair of electrodes, and a high-voltage transformer. Fusion can be observed with as little as 10 kV between the electrodes. 0.4575
Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively difficult to implement and requires a complex system of vacuum tubes, electrodes, and transformers. Fusion can be observed with as little as 100 kV between the electrodes. 0.2751
Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce heavy-ion fusion reactions. This method is relatively difficult to implem

12. Run the exact same zero-shot-classification as the previous question, but this time pass the argument ***multi_label=True.***

    What is the absolute difference between the sum of the 3 probabilities in the previous question (which uses Softmax ) ans the sum of the 3 probabilities in this question ( which uses independent Sigmoid)? 

In [26]:
res_sigmoid = classifier(prompt_text, candidate_labels, multi_label=True)
sum_sigmoid = sum(res_sigmoid["scores"])

In [28]:
diff = abs( 1 - sum_sigmoid )
print("Absolute difference:", round(diff, 4))

Absolute difference: 0.9995


13. Let's try Generative AI instead of Classification.

    Load a Small Language Model like ***google/fan-t5-small*** using the Hugging Face pipeline("text2text-generation"). Construct the following exact string for row index 0: ***"Question:\[prompt]. Is the correct answer A:\[A] or B:\[B]? Answer with just the letter A or B."
    Pass this string to the pipeline, setting ***max_new_tokens=5***. What is the exact string output returned by the model?

In [11]:
! pip install --upgrade transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 70.8 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 20.6 MB/s eta 0:00:00
  Attempting uninstall: safetensors
    Found existing installation: safetensors 0.7.0
    Uninstalling safetensors-0.7.0:
      Successfully uninstalled safetensors-0.7.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [12]:
from transformers import pipeline

In [13]:
generator = pipeline("text2text-generation", model="google/flan-t5-small")

prompt_text = dataset[0]['prompt']

input_str = f"Question:{prompt_text}. \
Options: A:{dataset[0]['A']} B:{dataset[0]['B']} \
Which is correct? Answer with A or B."

result = generator(input_str, max_new_tokens=5)

print(result[0]['generated_text'])


KeyError: "Unknown task text2text-generation, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'image-to-image', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'question-answering', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'visual-question-answering', 'vqa', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection', 'translation_XX_to_YY']"

### text2text-generation not found in google/flan-t5-small model